# Pancancer enrichment analysis step 2: Find enriched pathways
# Version 1: Using GSEApy to access Enrichr

In [1]:
import cptac
import cptac.utils as ut
import pandas as pd
import numpy as np
import gseapy as gp
import os
import datetime

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

STEP1_DIR = "step1_outputs"
STEP1_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP1_FILE_PATH = os.path.join(STEP1_DIR, STEP1_FILE)

STEP2_DIR = "step2_outputs_gseapy"
STEP2_FILE = f"urls_{TIME_START}_from_{STEP1_FILE.rsplit('.', maxsplit=1)[0]}.tsv"
STEP2_FILE_PATH = os.path.join(STEP2_DIR, STEP2_FILE)

if not os.path.isdir(STEP2_DIR):
    os.mkdir(STEP2_DIR)

In [3]:
# Read in the file from step 1
data = pd.read_csv(STEP1_FILE_PATH, sep='\t', index_col=0)

# In the omics data, make all increases +1 and decreases -1
# Because these are ratioed abundances, we can't compare magnitudes between
# genes--we can only compare positive or negative
data = data.assign(simplified_change=data["change"])

data.loc[data["change"] > 0, "simplified_change"] = 1
data.loc[data["change"] < 0, "simplified_change"] = -1
data.loc[data["change"] == 0, "simplified_change"] = 0

In [10]:
data.head()

,cancer_type,protein,change,P_val,adj_p,reject_null,protein_str,simplified_change
0,ccrcc,"('A1BG', 'NP_570602.2')",0.282928,0.0,0.0,True,A1BG,1.0
1,ccrcc,"('A1CF', 'NP_620310.1')",-0.551358,0.0,0.0,True,A1CF,-1.0
2,ccrcc,"('A2M', 'NP_000005.2')",0.595512,0.0,0.0,True,A2M,1.0
4,ccrcc,"('AAAS', 'NP_056480.1')",0.173579,0.0,0.0,True,AAAS,1.0
5,ccrcc,"('AACS', 'NP_076417.2')",-0.215967,0.0,0.0,True,AACS,-1.0


In [4]:
# For each cancer, find enriched pathways for genes that changed (either up or down)   
all_enrichments = pd.DataFrame()

for cancer_type in data["cancer_type"].unique():
    cancer_subset = data[data["cancer_type"] == cancer_type]
    changed_proteins = cancer_subset["protein_str"].tolist()
    enriched_pathways = gp.enrichr(
        gene_list=changed_proteins,
        description="changed_in_tumor",
        gene_sets="Reactome_2016",
        outdir=None)
    cancer_enriched = enriched_pathways.res2d.assign(cancer_type=cancer_type)
    all_enrichments = all_enrichments.append(cancer_enriched)

In [5]:
# Make a column of just pathway IDs
all_enrichments = all_enrichments.assign(pathway_id=all_enrichments["Term"].str.rsplit(" ", n=1, expand=True)[1])

# Make a column of overlap proportions
overlaps = all_enrichments["Overlap"].str.split("/", expand=True).astype("int")
overlap_props = overlaps[0] / overlaps[1]
all_enrichments = all_enrichments.assign(overlap_prop=overlap_props)

In [6]:
all_enrichments.head()

,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Gene_set,cancer_type,pathway_id,overlap_prop
0,Metabolism Homo sapiens R-HSA-1430728,1305/1908,1.094902e-110,1.675200e-107,0,0,1.545328,391.267360,NUP107;PSMD8;NDST1;IL4I1;LIPE;PSMD6;AS3MT;NSDH...,Reactome_2016,ccrcc,R-HSA-1430728,0.683962
1,Infectious disease Homo sapiens R-HSA-5663205,281/348,4.565186e-45,3.492367e-42,0,0,1.824382,186.265468,RPL4;RPL5;CCNK;RPL3;NUP107;RPL32;CCNH;RPL31;RP...,Reactome_2016,ccrcc,R-HSA-5663205,0.807471
2,Metabolism of proteins Homo sapiens R-HSA-392499,693/1074,5.698366e-43,2.906167e-40,0,0,1.457866,141.808033,RPL4;MDC1;RPL5;PGAP1;NUP107;RPL3;RPL32;RPL31;R...,Reactome_2016,ccrcc,R-HSA-392499,0.645251
3,Disease Homo sapiens R-HSA-1643685,491/725,1.595132e-38,6.101381e-36,0,0,1.530143,133.170316,RPL4;RPL5;RPL3;NUP107;MAML2;RPL32;ZFYVE9;MAML1...,Reactome_2016,ccrcc,R-HSA-1643685,0.677241
4,Metabolism of amino acids and derivatives Homo...,258/335,7.060858e-35,2.160622e-32,0,0,1.740057,136.830973,RPL4;RPL5;MTRR;RPL3;RPL32;RPL31;GLDC;RPL34;RPL...,Reactome_2016,ccrcc,R-HSA-71291,0.770149


In [7]:
# Make a table with a list of all pathways, and:
# - The number of cancer types for which that cancer type is enriched
# - The average of the adjusted p values for that pathway
# - The average overlap proportion for that pathway
enrichment_summary = all_enrichments[["pathway_id", "Term"]].drop_duplicates(keep="first")

times_enriched = enrichment_summary["pathway_id"].apply(
    lambda x: all_enrichments[all_enrichments["pathway_id"] == x].shape[0])

avg_p_vals = enrichment_summary["pathway_id"].apply(
    lambda x: all_enrichments.loc[all_enrichments["pathway_id"] == x, "Adjusted P-value"].mean())

avg_overlap_props = enrichment_summary["pathway_id"].apply(
    lambda x: all_enrichments.loc[all_enrichments["pathway_id"] == x, "overlap_prop"].mean())

enrichment_summary = enrichment_summary.assign(
    times_enriched=times_enriched,
    avg_adj_p=avg_p_vals,
    avg_overlap_prop=avg_overlap_props)

enrichment_summary = enrichment_summary.sort_values(by=["times_enriched", "avg_overlap_prop"], ascending=[False, False])
enrichment_summary = enrichment_summary.reset_index(drop=True)

In [8]:
enrichment_summary.head()

,pathway_id,Term,times_enriched,avg_adj_p,avg_overlap_prop
0,R-HSA-174577,Activation of C3 and C5 Homo sapiens R-HSA-174577,8,0.032375,0.958333
1,R-HSA-166665,Terminal pathway of complement Homo sapiens R-...,8,0.114946,0.921875
2,R-HSA-174411,Polymerase switching on the C-strand of the te...,8,0.012370,0.901786
3,R-HSA-69091,Polymerase switching Homo sapiens R-HSA-69091,8,0.012351,0.901786
4,R-HSA-69109,Leading Strand Synthesis Homo sapiens R-HSA-69109,8,0.012332,0.901786


In [1]:
# Visualize selected pathways (this will show which genes are up or down)
to_viz = enrichment_summary[["pathway_id", "Term"]].iloc[0:4, :]

cancer_types_list = []
terms_list = []
pathways_list = []
urls_list = []

for i in range(to_viz.index.size):
    pathway = to_viz.iloc[i, 0]
    term = to_viz.iloc[i, 1].rsplit(" ", maxsplit=1)[0]
    for cancer_type in data["cancer_type"].unique():
        
        omics = data.loc[data["cancer_type"] == cancer_type, "simplified_change"].\
            set_index("protein_str").\
            rename(columns={"simplified_change": f"{cancer_type}_simp_change"})
        
        url = ut.reactome_pathway_overlay(
            pathway=pathway,
            df=omics,
            open_browser=False)
        
        cancer_types_list.append(cancer_type)
        terms_list.append(term)
        pathways_list.append(pathway)
        urls_list.append(url)

SyntaxError: invalid syntax (<ipython-input-1-806afe19e516>, line 16)

In [12]:
# Create a dataframe of the visualization URLs
urls_df = pd.DataFrame({
    "cancer_type": cancer_types_list,
    "pathway_name": terms_list,
    "pathway_id": pathways_list,
    "url": urls_list})

In [13]:
# Print the dataframe in such a way that the links are clickable
urls_df.style.format('<a href="{0}">{0}</a>', subset="url")

,cancer_type,pathway_name,pathway_id,url
0,ccrcc,Activation of C3 and C5 Homo sapiens,R-HSA-174577,https://reactome.org/PathwayBrowser/#/R-HSA-174577&DTAB=AN&ANALYSIS=MjAyMDA1MzAwMjEwNTJfMTU3NTA%3D
1,colon,Activation of C3 and C5 Homo sapiens,R-HSA-174577,https://reactome.org/PathwayBrowser/#/R-HSA-174577&DTAB=AN&ANALYSIS=MjAyMDA1MzAwMjEwNTZfMTU3NTE%3D
2,endometrial,Activation of C3 and C5 Homo sapiens,R-HSA-174577,https://reactome.org/PathwayBrowser/#/R-HSA-174577&DTAB=AN&ANALYSIS=MjAyMDA1MzAwMjEwNTlfMTU3NTI%3D
3,gbm,Activation of C3 and C5 Homo sapiens,R-HSA-174577,https://reactome.org/PathwayBrowser/#/R-HSA-174577&DTAB=AN&ANALYSIS=MjAyMDA1MzAwMjExMDJfMTU3NTM%3D
4,hnscc,Activation of C3 and C5 Homo sapiens,R-HSA-174577,https://reactome.org/PathwayBrowser/#/R-HSA-174577&DTAB=AN&ANALYSIS=MjAyMDA1MzAwMjExMDVfMTU3NTQ%3D
5,lscc,Activation of C3 and C5 Homo sapiens,R-HSA-174577,https://reactome.org/PathwayBrowser/#/R-HSA-174577&DTAB=AN&ANALYSIS=MjAyMDA1MzAwMjExMDdfMTU3NTU%3D
6,luad,Activation of C3 and C5 Homo sapiens,R-HSA-174577,https://reactome.org/PathwayBrowser/#/R-HSA-174577&DTAB=AN&ANALYSIS=MjAyMDA1MzAwMjExMTBfMTU3NTY%3D
7,ovarian,Activation of C3 and C5 Homo sapiens,R-HSA-174577,https://reactome.org/PathwayBrowser/#/R-HSA-174577&DTAB=AN&ANALYSIS=MjAyMDA1MzAwMjExMTJfMTU3NTc%3D
8,ccrcc,Terminal pathway of complement Homo sapiens,R-HSA-166665,https://reactome.org/PathwayBrowser/#/R-HSA-166665&DTAB=AN&ANALYSIS=MjAyMDA1MzAwMjEwNTJfMTU3NTA%3D
9,colon,Terminal pathway of complement Homo sapiens,R-HSA-166665,https://reactome.org/PathwayBrowser/#/R-HSA-166665&DTAB=AN&ANALYSIS=MjAyMDA1MzAwMjEwNTZfMTU3NTE%3D


In [14]:
# Save the results
urls_df.to_csv(STEP2_FILE_PATH, sep='\t')